# 误差反向传播法

In [1]:
import numpy as np
import sys
sys.path.insert(0, r"D:\.YOLO_FPN_FPGA_Accelerator_Paper\.DeepLearning\fish-book-practices-main\fish-book-practices-main")

### 简单层的实现

In [5]:
class MulLayer:
    """
    乘法层
    """
    def __init__(self):
        self.x = None
        self.y = None
        
    def forward(self, x, y):
        self.x = x
        self.y = y
        out = x * y
        
        return out
    
    def backward(self, dout):
        dx = dout * self.y
        dy = dout * self.x
        
        return dx, dy

class AddLayer:
    """
    加法层
    """
    def __init__(self):
        pass
    
    def forward(self, x, y):
        out = x + y
        
        return out
    
    def backward(self, dout):
        dx = dout * 1
        dy = dout * 1
        
        return dx, dy


#### 实现"购买2个苹果和3个橘子"


![购买2个苹果和3个橘子](../screen-short/5-17.png)

### 激活函数层实现

In [4]:
# ReLU
class ReluLayer:
    """
    ReLU层
    """
    def __init__(self):
        self.mask = None
        
    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0
        
        return out
    
    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout
        
        return dx

In [5]:
# Sigmoid
class SigmoidLayer:
    """
    Sigmoid层
    """
    def __init__(self):
        self.out = None
        
    def forward(self, x):
        self.out = 1 / (1 + np.exp(-x))
        return self.out
    
    def backward(self, dout):
        dx = dout * (1.0 - self.out) * self.out
        return dx

In [6]:
# Softmax with Cross Entropy Loss
class SoftmaxWithLoss:
    """
    Softmax with Cross Entropy Loss
    """
    def __init__(self):
        self.y = None
        self.t = None
        
    def forward(self, x, t):
        self.t = t
        exp_x = np.exp(x - np.max(x))  # 防止溢出
        self.y = exp_x / np.sum(exp_x)
        
        loss = -np.sum(t * np.log(self.y + 1e-7))  # 加上小常数防止log(0)
        
        return loss
    
    def backward(self):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size
        
        return dx

In [7]:
# Affine Layer
class AffineLayer:
    """
    仿射层
    """
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None
        
    def forward(self, x):
        self.x = x
        out = np.dot(x, self.W) + self.b
        
        return out
    
    def backward(self, dout):
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        dx = np.dot(dout, self.W.T)
        
        return dx

In [6]:
# 运行案例：购买2个苹果和3个橘子，并计算各变量对最终价格的影响
# 计算图结构：
# apple * apple_num -> apple_price
# orange * orange_num -> orange_price
# apple_price + orange_price -> fruit_price
# fruit_price * tax -> price

apple = 100
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1

mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_fruit_layer = AddLayer()
mul_tax_layer = MulLayer()

# forward
apple_price = mul_apple_layer.forward(apple, apple_num)
orange_price = mul_orange_layer.forward(orange, orange_num)
fruit_price = add_fruit_layer.forward(apple_price, orange_price)
price = mul_tax_layer.forward(fruit_price, tax)

print("前向传播结果")
print("apple_price =", apple_price)
print("orange_price =", orange_price)
print("fruit_price =", fruit_price)
print("price =", price)

# backward
dprice = 1
dfruit_price, dtax = mul_tax_layer.backward(dprice)
dapple_price, dorange_price = add_fruit_layer.backward(dfruit_price)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)

print("\n反向传播结果")
print("dapple =", dapple)
print("dapple_num =", dapple_num)
print("dorange =", dorange)
print("dorange_num =", dorange_num)
print("dtax =", dtax)

前向传播结果
apple_price = 200
orange_price = 450
fruit_price = 650
price = 715.0000000000001

反向传播结果
dapple = 2.2
dapple_num = 110.00000000000001
dorange = 3.3000000000000003
dorange_num = 165.0
dtax = 650
